# Augmented LLMs as Agents

AI agents are artificial intelligence systems that interacts with its environment. We will be primarily interested in LLM-based agents that uses a core LLM system for **reasoning**, **planning**, and **tool calling**. In practice, this also involve developing tools that are designed to interface with LLM-based agents. The foundational framework of reasoning (via [chain-of-thought](@fig-cot-prompting)) and task-specific actions for LLMs is explored in the **ReAct paper** [@ReAct2023] (@fig-react-paper):

> [We] explore the use of LLMs to generate both reasoning traces and task-specific actions in an interleaved manner, allowing for greater synergy between the two: reasoning traces help the model induce, track, and update action plans as well as handle exceptions, while actions allow it to interface with external sources, such as knowledge bases or environments, to gather additional information. [...] On two interactive decision making benchmarks (ALFWorld and WebShop), ReAct outperforms imitation and reinforcement learning methods by an absolute success rate of 34% and 10% respectively, while being prompted with only one or two in-context examples.

![**ReAct vs CoT only and Act only methods.** [ALFWorld](https://alfworld.github.io/) is a text-based environment (similar to text adventure RPGs in the old days) designed for agents to reason and learn high-level policies. [HotpotQA](https://hotpotqa.github.io/) is a question-answering dataset containing multi-hop (i.e. requiring resolving intermediate steps to get to the final answer) questions in natural language, with strong supervision for supporting facts.](./img/react-paper.png){#fig-react-paper}

![**Chain-of-thought** (CoT) prompting.](./img/zero-cot.png){#fig-cot-prompting}

<br>

## LLM inference via APIs

### OpenAI API client

To explore building effective AI agents, we can start with pure Python and LLM APIs. In particular, we use the [Python SDK](https://github.com/openai/openai-python/tree/main) for the [OpenAI API](https://platform.openai.com/docs/api-reference/introduction). First, we need to load the API key in the environmental variables. The client expects the environmental variable `OPENAI_API_KEY` which we can load from the `.env` file. This is easy enough to implement:

In [1]:
import inspect
from notebooks.utils import load_dotenv, print
print(inspect.getsource(load_dotenv))

def load_dotenv(verbose=False):
    with open(".env") as f:
        for line in f.readlines():
            k, v = line.split("=")
            os.environ[k] = v.strip().strip('"')
            if verbose:
                print(f"Loaded env variable: {k}")



In [2]:
load_dotenv(verbose=True)

Loaded env variable: OPENAI_API_KEY
Loaded env variable: GROQ_API_KEY


Then the API key is automatically read by the **client**:

In [3]:
from openai import OpenAI

client = OpenAI()

completion = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role": "system", 
            "content": "You're a helpful assistant."},
        {
            "role": "user",
            "content": "Tell me about the history of GPT in a single paragraph.",
        },
    ],
)

response_text = completion.choices[0].message.content

:::{.callout-note}
We can have **multiple completions** of the same prompt. This allows choosing between the responses.
For example, we can set `temperature=0.9` to get more varied outputs, so that choosing becomes nontrivial. 
By default only 1 completion by default, hence `[0]`.
:::

In [4]:
print(response_text, wrap=True)

GPT, or Generative Pre-trained Transformer, is a series of language models
developed by OpenAI. The journey began with the release of GPT in 2018, a model
built on the transformer architecture, which was introduced by Vaswani et al. in
2017. This was followed by GPT-2 in 2019, which gained attention for its larger
scale and enhanced text generation capabilities, leading OpenAI to initially
withhold its full version due to concerns about misuse. In 2020, GPT-3 was
released, boasting 175 billion parameters and showcasing remarkable abilities in
understanding and generating human-like text, prompting widespread usage in
various applications. In March 2023, OpenAI announced GPT-4, marking further
advancements in language understanding. These models illustrate the rapid
evolution and growing impact of AI in natural language processing.


Entire model response:

In [5]:
from pprint import pprint
pprint(completion.model_dump())

{'choices': [{'finish_reason': 'stop',
              'index': 0,
              'logprobs': None,
              'message': {'annotations': [],
                          'audio': None,
                          'content': 'GPT, or Generative Pre-trained '
                                     'Transformer, is a series of language '
                                     'models developed by OpenAI. The journey '
                                     'began with the release of GPT in 2018, a '
                                     'model built on the transformer '
                                     'architecture, which was introduced by '
                                     'Vaswani et al. in 2017. This was '
                                     'followed by GPT-2 in 2019, which gained '
                                     'attention for its larger scale and '
                                     'enhanced text generation capabilities, '
                                     'leading OpenAI

:::{.callout-note}
We are using the **chat completions** API where an autoregressive process that's running under the hood. Here the prompt to be completed is:

```python
messages=[
    {"role": "system", "content": "You are a poetic but terse assistant."},   # prompt
    {"role": "user", "content": "What is the color of the sky?"}              # prompt
]
```

And the completion is given by the API's output:

```python
{
  "role": "assistant",
  "content": "The sky's color shifts from azure to amber, a canvas for sun's daily journey."
}
```

**Remark.** The generated response is statistically the most likely continuation of the prompt text sequence. 
:::

### Structured outputs

[Structured outputs](https://platform.openai.com/docs/guides/structured-outputs?api-mode=responses) is a feature that ensures that a model generates responses that adhere to a supplied **schema** (e.g. a Pydantic model). As such, the output can then be parsed using the same Pydantic model. Structured outputs makes prompting significantly simpler: no more need for strongly worded prompts to achieve consistent formatting, no explicitly having to retry incorrectly formatted responses, or having invalid hallucinated values (can specify **enums**).

In [6]:
# https://platform.openai.com/docs/guides/structured-outputs?api-mode=responses
from openai import OpenAI
from pydantic import BaseModel

client = OpenAI()

class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

completion = client.chat.completions.parse(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "Extract the event information."},
        {
            "role": "user",
            "content": "Alice and Bob are going to a science fair on Friday.",
        },
    ],
    response_format=CalendarEvent,
)

event = completion.choices[0].message.parsed
event

CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

## LLM augmentations: goals, tools, & memory

The foundational building block of agentic systems is a core language model service interfaces with **goals**, **tools**, and **memory** modules. Since current models are able to effectively plan and reason, this greatly improves the problem solving ability of such systems. Moreover, augmentation is necessary for LLMs to reason about data outside of their training data (i.e. data that is outdated relative to the training cutoff). These augmentations also allow the agentic module to [interact with its environment]{.underline}.

![A diagram of **one request-response cycle** with a core LLM service augmented with goals, tools, and memory capabilities. Interaction with environment occurs in step (3.2).](./img/augmented-llm.png){#fig-augmentedllm}

:::{.callout-note}
Many descriptions of agentic systems explicitly include **retrieval** as a core module (e.g. [Agentic RAG](https://langchain-ai.github.io/langgraph/tutorials/rag/langgraph_agentic_rag/#agentic-rag)). But retrieval can be part of the tools or memory module so we just abstract over it. Instead, we introduce a *Goals* module &mdash; less common in standard formulations &mdash; which highlights the significant role of prompt engineering and spec-driven development in guiding system behavior.
:::

### Function calling

Also known as **tool calling**. Function calling give models access to external tools and data that they can use to respond to prompts. Since LLMs *only* consume and generate text, they cannot actually execute functions. Instead, the main program listens to the LLM hallucinate and executes the commands based on that (@fig-brainvat).

![**LLM as brain in a vat.** The LLM as core reasoning module thinking of what function to execute with what arguments without having the capability to execute them. We provide a separate process and environment for running the code. [Source](https://en.wikipedia.org/wiki/Brain_in_a_vat) ](./img/brain-vat.png){#fig-brainvat width=70%}

**Task.** To demonstrate tool calling, we develop a system of querying the weather in [Quisao](https://www.philatlas.com/luzon/r04a/rizal/pililla/quisao.html) using natural language. It essentially builds on the OpenAI **chat completions API** (see @fig-chat-completions). Basically, we want the agent to call the following core API:

In [7]:
import json
import requests

def get_weather(latitude, longitude):
    """
    Get current weather data for provided coordinates with units:
    temperature (celsius), wind speed (kph), & precipitation (mm).
    """
    response = requests.get((
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&"
        "current=temperature_2m,wind_speed_10m,relative_humidity_2m,precipitation,precipitation_probability"
    ))
    data = response.json()
    return data["current"]


get_weather(latitude=14.4779, longitude=121.3214)  # true coordinates

{'time': '2025-09-03T12:00',
 'interval': 900,
 'temperature_2m': 28.6,
 'wind_speed_10m': 10.7,
 'relative_humidity_2m': 84,
 'precipitation': 0.2,
 'precipitation_probability': 53}

**Tool definition.** For the LLM to understand a specific tool, we have to define a schema that informs the model of what the tool does and its expected (required and optional) arguments. The following is the function definition for `get_weather`:

In [8]:
tools = [
    {
        "type": "function", # <1>
        "function": {
            "name": "get_weather",  # <2>
            "description": "Get current weather data for provided coordinates with units: temperature (celsius), wind speed (kph), & precipitation (mm).",    # <3>
            "parameters": {
                "type": "object", # <4>
                "properties": { # <5>
                    "latitude": {
                        "type": "number"
                    },
                    "longitude": {
                        "type": "number"
                    },
                },
                "required": ["latitude", "longitude"],
                "additionalProperties": False,  # <6>
            },
            "strict": True, # <7>
        },
    }
]

1. Should always be function.
2. Function's name (i.e. `get_weather`).
3. Usually just the docstring. Should describe when and how to use the function.
4. Parameters are naturally JSON objects.
5. List of arguments. Clearly the two arguments are required.
6. Part of JSON schema that determines whether extra fields are valid or not.
7. Not part of JSON schema, but OpenAI function calling option that *guides* the model to strictly follow the schema (i.e. not improvise). Setting `additionalProperties` to `False` and `strict` to `True` works together to ensure that the model generates the correct parameters schema.

:::{.callout-tip} 
Because `parameters` is defined by a JSON schema, you can leverage many of its rich features like property types, enums, descriptions, nested objects, and so on. For example, we can have:

```
"properties": {
    "unit": {
        "type": "string",
        "enum": ["celsius", "fahrenheit"],
        "description": "Unit of measure for temperature."
    }
}
```
:::

**Agent definition.** We define messages outside of the agent to track chat history.

In [9]:
system_prompt = "You are a helpful weather assistant."
messages = [{"role": "system", "content": system_prompt}]

def get_weather_agent(messages: list, update=True):
    """Agent with access to `get_weather` API. Responds with tool calls."""
    
    completion = client.chat.completions.create(
        model="gpt-4.1",
        messages=messages,
        tools=tools,
    )

    out = completion.choices[0].message
    if update:
        messages.append(out.model_dump())

    return out


# query step. expects tool_calls
query = "What's the weather like in Quisao, Pililla, Rizal right now?"
messages.append({"role": "user", "content": query})
response = get_weather_agent(messages)

pprint(response.model_dump())

{'annotations': [],
 'audio': None,
 'content': None,
 'function_call': None,
 'refusal': None,
 'role': 'assistant',
 'tool_calls': [{'function': {'arguments': '{"latitude":14.4944,"longitude":121.3239}',
                              'name': 'get_weather'},
                 'id': 'call_4QeMnQu5xVJpJZxeeIHWuejY',
                 'type': 'function'}]}


It's impressive that fairly accurate coordinates were obtained without using web search (i.e. these facts were inferred from the model weights). Observe that whenever we have `tools`, the model responds with only `tool_calls` as nonnull (e.g. `content` is empty) with role `assistant`.  

**Function calls.** We will iterate over tool calls and process them separately. Each function call and their results are then logged in the message history. This explains why we defined `messages` outside of the API call unlike the usual setup .

In [10]:
def call_function(name, args):
    fn = {
        "get_weather": get_weather  
    }
    return fn[name](**args)


for tool_call in response.tool_calls:
    args = json.loads(tool_call.function.arguments)
    name = tool_call.function.name
    tool_output = call_function(name, args)
    
    messages.append({    
        "role": "tool",     # <1>
        "tool_call_id": tool_call.id, 
        "content": json.dumps(tool_output)
    })

1. Tool calls are logged with role `tool`. 

Message history appended with actual API output:

In [11]:
pprint(messages)

[{'content': 'You are a helpful weather assistant.', 'role': 'system'},
 {'content': "What's the weather like in Quisao, Pililla, Rizal right now?",
  'role': 'user'},
 {'annotations': [],
  'audio': None,
  'content': None,
  'function_call': None,
  'refusal': None,
  'role': 'assistant',
  'tool_calls': [{'function': {'arguments': '{"latitude":14.4944,"longitude":121.3239}',
                               'name': 'get_weather'},
                  'id': 'call_4QeMnQu5xVJpJZxeeIHWuejY',
                  'type': 'function'}]},
 {'content': '{"time": "2025-09-03T12:00", "interval": 900, "temperature_2m": '
             '28.6, "wind_speed_10m": 10.7, "relative_humidity_2m": 84, '
             '"precipitation": 0.2, "precipitation_probability": 53}',
  'role': 'tool',
  'tool_call_id': 'call_4QeMnQu5xVJpJZxeeIHWuejY'}]


:::{.callout-note}
Chat completion API calls have stateless single request-response cycles which gives the user straightforward control over the **message history**. This can be seen in the above example where we manually manage message history with tool calls declaration as well as actual function outputs.
:::

**Report agent.** Next, we pass this thread to another agent (possibly to a different, more specialized model) which will process the outputs of the tool calls along with earlier chat messages. To take advantage of structured outputs we again define a response format. Here we use Pydantic `Field` with a description that helps the LLM.

In [12]:
from pydantic import Field

class WeatherReport(BaseModel):
    response: str = Field(description="A natural language response to the user's question.")
    temperature: float = Field(description="Current temperature in celsius for the given location.")


def weather_report_agent(messages: list, update=True):
    completion = client.chat.completions.parse(
        model="gpt-4o",
        messages=messages,
        tools=tools,    # <!>
        response_format=WeatherReport,
    )
    
    out = completion.choices[0].message
    if update:
        messages.append(out.model_dump())
    
    return out

:::{.callout-caution}
The aggregator also needs access to tools for it to understand the context of each tool call!
:::

**Final output.** Note that the units correctly identified from the `get_weather` docstring:

In [13]:
response = weather_report_agent(messages)
weather_report = response.parsed
print("temp:", weather_report.temperature, "\n")
print(weather_report.response, wrap=True)

temp: 28.6 

The current weather in Quisao, Pililla, Rizal is warm with a temperature of
28.6°C. The humidity is relatively high at 84%, and there is a light breeze with
wind speeds of 10.7 kph. There's also a chance of light precipitation, with
recorded precipitation of 0.2 mm.


![Creating a weather report using the chat completions API with LLM agents.](./img/chat-completions-api.png){#fig-chat-completions}

### Memory and retrieval

The following is a toy example of an agent that reads and writes to an external data source. One characteristic of retrieval systems is that the entire process is **stateless**, e.g. it cannot learn from interactions. Retrieval systems generally involve queries to an external data source, then adding the response to the current model context.

For the following example, the LLM also writes to the same memory store hence affecting future generation states. It follows that this system is **stateful**. We can think of the external memory store as the **long-term memory** of the system. On the other hand, LLMs naturally have **short-term memory** in the form of its context. The architecture is shown in @fig-retrieval-system.

![The LLM expresses the intent to write to the memory store via the structured output. Then, it is up to the main program to perform the actual writing. This allows hooks like [guardrails](https://cookbook.openai.com/examples/how_to_use_guardrails) to be applied before executing the function. Note that the retrieval happens prior to LLM processing. It would be nice to have the LLM read the entire filestore but this becomes more expensive as the memory store grows. In practice, information retrieval techniques such as TF-IDF and embedding similarity can be used.
](./img/retrieval-system.png){#fig-retrieval-system}

**Memory store.** Memories are saved as dictionaries `{"tag": <tag>, "fact": <fact>}`. The following objects work around this definition. A retrieval function is defined as a method of the memory store class which gets relevant memory items based on *keyword search*. Hence, we have the following function for extracting keywords:

In [14]:
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words("english"))

def tokenize(text):
    """Tokenize into words then remove stopwords."""
    tokens = word_tokenize(text.lower())
    filtered_tokens = [word for word in tokens if word not in stop_words and word not in string.punctuation]
    return filtered_tokens

text = "Punk is a pre-trained unsupervised machine learning model for tokenization. It's one of the most crucial and widely used components in the NLTK library."
print("Original:", text)
print("Filtered:", tokenize(text))

Original: Punk is a pre-trained unsupervised machine learning model for tokenization. It's one of the most crucial and widely used components in the NLTK library.
Filtered: ['punk', 'pre-trained', 'unsupervised', 'machine', 'learning', 'model', 'tokenization', "'s", 'one', 'crucial', 'widely', 'used', 'components', 'nltk', 'library']


In [15]:
import json
from typing import List
from openai import OpenAI
from pydantic import BaseModel


class MemoryItem(BaseModel):
    tag: str
    fact: str
    reason: str

class MemoryResponse(BaseModel):
    items: List[MemoryItem]


class MemoryStore:
    def __init__(self, path="memory.json"):
        """Load memory from JSON file in local path."""
        self.path = path
        self.data = []
        self.tags = set()
        self.load()

    def load(self):
        try:
            self.data = json.load(open(self.path))
        except FileNotFoundError:
            self.reset()

    def save(self):
        with open(self.path, "w") as f:
            json.dump(self.data, f, indent=2)

    def reset(self):
        self.data = []
        self.tags = set()
        self.save()
    
    def add(self, item: MemoryItem):
        tag, fact = item.tag, item.fact
        self.tags.add(tag)
        self.data.append({"tag": tag, "fact": fact})

    def __len__(self):
        return len(self.data)
    
    def retrieve(self, query: str, topk: int = 3) -> List[dict]:
        """Simple keyword-based retrieval."""
        
        query_words = tokenize(query)
        retrieved = []
        
        for memory in reversed(self.data):  # <1>
            tag, fact = memory["tag"], memory["fact"]
            fact = ' '.join(tokenize(fact))
            memory_text = f"{tag} {fact}".lower()

            for word in query_words:
                if word in memory_text: # <2>
                    retrieved.append(memory)
                    break
            
            if len(retrieved) == topk:
                break
        
        return retrieved


client = OpenAI()
mem = MemoryStore()

1. More recent = more relevant.
2. Substring check. e.g. `'commute' in 'commute_experience'` evaluates to `True`.

Next, we define the **generation step** and the **write step**:

In [16]:
prompt_template = lambda memories, tags: f"""
You are an assistant that processes daily user logs. For each log, extract a concise, 
factual summary of what happened. Each summary should be atomic, standalone, and 
likely useful for future interactions. Assign a relevant `tag` to each summary (`fact`) 
before saving it to memory. 

The following are relevant entries (based on the current input) in the Memory Store:
{memories}

The following are the current tags:
{tags}

**GUIDELINES:**

1. **EXTRACT ATOMIC FACTS:**
    - Break down information into the smallest meaningful, self-contained units.
    - **Good**: "User's favorite programmer is Jon Blow."
    - **Bad**: "User mentioned their favorite programmer is Jon Blow who is a famous game programmer" (This has two facts.)
    - The `fact` must be a concise, direct paraphrase of the fact. Remove conversational fluff.
    - **Good Info:** "User's favorite city is Tokyo"
    - **Bad Info:** "The user stated that if they had to pick a favorite city, they think it would be Tokyo."

2.  **TAG EFFECTIVELY:**
    - **Format:** Prefer generic, descriptive tags in `snake_case`.
    - **Simple:** Prefer simple tags. Choose `commute` is better than `commute_experience`.
    - **Reuse:** Strongly prefer existing tags. Create a new tag only if necessary.
    - An example: For "I really enjoy hiking in the Alps every summer," a good tag is `hobby` or `outdoor_activity`.
    
4.  **EVALUATE & DECIDE:**
    - It is acceptable to save zero logs from an input if nothing is meaningfully new or relevant.
    - Save multiple logs if the user provides multiple distinct pieces of information.
    - Each memory item should make sense on its own. There should be no dependence between separate logs.
"""


def capture_memorable_facts(user_log: str, topk: int=3) -> MemoryResponse:
    """Generate memorable facts from log and write them to memory."""

    # generate memorable facts based on relevant items
    retrieved_memories = mem.retrieve(user_log, topk)
    current_tags = list(mem.tags)
    system_prompt = prompt_template(retrieved_memories, current_tags)

    response = client.chat.completions.parse(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_log},
        ],
        response_format=MemoryResponse,
    )
    
    # save items to memory store
    out = response.choices[0].message.parsed
    for item in out.items:
        mem.add(item)

    mem.save()
    return out

Examples:

In [17]:
import pandas as pd

logs = [
    "Woke up later than usual because I forgot to set an alarm, rushed through a quick shower, skipped coffee, and still managed to leave for work on time.",
    "Traffic was unusually light, but halfway through I realized I left my ID at home, debated turning back, and decided to just explain at the office front desk.",
    "Took the train, found no seats since it was packed, but struck up a short conversation with a stranger about the book they were reading while we both stood.",
    "Stopped by the bakery, picked up bread for the team, and noticed they had a new seasonal pastry that tempted me but I decided to pass.",
    "Opened my email first thing at the office, skimmed through a pile of routine messages, flagged one urgent client request, and forwarded it to the team lead.",
    "Woah! While crossing the street a man suddenly darted into traffic, cars honked, everyone gasped, and I stood frozen for a moment before hurrying away still shaken.",
    "Listened to a podcast while walking to the subway, half distracted by construction noise on the street, and made a mental note to check out the book they recommended.",
    "Grabbed a pen from my drawer because mine ran out of ink, ended up reorganizing the entire drawer, and discovered an old sticky note with a reminder I had long forgotten."
]

items = []
for log in logs:
    memory_items = capture_memorable_facts(log).items
    for item in memory_items:
        d = item.model_dump()
        d["text"] = log
        items.append(d)

df_resp = pd.DataFrame(items)
mem.reset()

The agent decides whether to reuse a tag or create a new one based on the data:

In [18]:
#| code-fold: true
import warnings
warnings.simplefilter("ignore")
pd.set_option('display.max_colwidth', None)

print(f"({len(logs)} total logs, {len(df_resp)} facts saved, {len(df_resp.tag.unique())} tags)")
print("tags:")
pprint(list(df_resp.tag.unique()))
df_resp[["tag", "fact", "text", "reason"]]

(8 total logs, 10 facts saved, 8 tags)
tags:
['commute',
 'food_purchase',
 'work_task',
 'unexpected_incident',
 'media_consumption',
 'book_interest',
 'workspace_organization',
 'personal_discovery']


,tag,fact,text,reason
0,commute,"User managed to leave for work on time despite waking up late, skipping coffee, and rushing through a shower.","Woke up later than usual because I forgot to set an alarm, rushed through a quick shower, skipped coffee, and still managed to leave for work on time.",This aligns with existing commute experiences and contributes to understanding daily schedules.
1,commute,User forgot their ID at home but decided to explain at the office front desk instead of turning back.,"Traffic was unusually light, but halfway through I realized I left my ID at home, debated turning back, and decided to just explain at the office front desk.","This fact complements the existing information about the commute, noting the decision-making process and potential impact."
2,food_purchase,User picked up bread for the team at the bakery.,"Stopped by the bakery, picked up bread for the team, and noticed they had a new seasonal pastry that tempted me but I decided to pass.","This log details a specific purchase made by the user, which may be relevant for future preferences or routines."
3,work_task,User opened their email first thing at the office and forwarded an urgent client request to the team lead.,"Opened my email first thing at the office, skimmed through a pile of routine messages, flagged one urgent client request, and forwarded it to the team lead.",The log provides a summary of a specific task conducted by the user as part of their work routine.
4,unexpected_incident,"User witnessed a man dart into traffic, causing cars to honk and bystanders to gasp, which left them shaken.","Woah! While crossing the street a man suddenly darted into traffic, cars honked, everyone gasped, and I stood frozen for a moment before hurrying away still shaken.",This is a distinct and memorable experience regarding an unexpected and alarming incident.
5,media_consumption,User listened to a podcast while walking to the subway.,"Listened to a podcast while walking to the subway, half distracted by construction noise on the street, and made a mental note to check out the book they recommended.","This indicates what the user was doing during their walk, related to media consumption."
6,commute,User was half distracted by construction noise on the street while walking to the subway.,"Listened to a podcast while walking to the subway, half distracted by construction noise on the street, and made a mental note to check out the book they recommended.",This describes part of the user's experience during their commute.
7,book_interest,User made a mental note to check out a book recommended in a podcast.,"Listened to a podcast while walking to the subway, half distracted by construction noise on the street, and made a mental note to check out the book they recommended.",This indicates a new interest or intention to explore a book.
8,workspace_organization,User reorganized their drawer after retrieving a pen.,"Grabbed a pen from my drawer because mine ran out of ink, ended up reorganizing the entire drawer, and discovered an old sticky note with a reminder I had long forgotten.",Recording about workspace organization may provide context for future tasks or preferences.
9,personal_discovery,User discovered an old sticky note with a forgotten reminder in their drawer.,"Grabbed a pen from my drawer because mine ran out of ink, ended up reorganizing the entire drawer, and discovered an old sticky note with a reminder I had long forgotten.",Finding a forgotten reminder may impact user's future decisions or tasks.
